In [1]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import sys
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")

from src.model import get_resnet_model
from src.dataset import BackdooredDataset
from src.backdoor import gaussian_noise_static_trigger

In [2]:
class Config:
    BATCH_SIZE = 128


CLASSES = [
    "plane",
    "car",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

In [3]:
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU: NVIDIA GeForce RTX 3070


In [4]:
def get_model():
    # model = get_resnet_model(100)
    # checkpoint_tl_cifar100 = torch.load(
    #     "../weights/weights-gauss-static-tf-cifar100.pth", map_location=DEVICE
    # )
    # model.load_state_dict(checkpoint_tl_cifar100["model_state_dict"])

    # model.fc = nn.Linear(model.fc.in_features, 10)
    #
    # checkpoint_cifar10 = torch.load(
    #     "../weights/weights-gaussian-noise-static.pth", map_location=DEVICE
    # )
    #
    # for k in checkpoint_cifar10["model_state_dict"].keys():
    #     print(k)
    #
    # model.fc.weight.data.copy_(checkpoint_cifar10["model_state_dict"]["fc.weight"])
    # model.fc.bias.data.copy_(checkpoint_cifar10["model_state_dict"]["fc.bias"])

    model = get_resnet_model(10)
    checkpoint = torch.load(
        "../weights/weights-after-tf-linear-probes-cifar10.pth", map_location=DEVICE
    )
    model.load_state_dict(checkpoint["model_state_dict"])

    model.to(DEVICE)

    return model

In [5]:
def get_data_loaders():
    transform_test = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]
            ),
        ]
    )

    test_dataset = BackdooredDataset(
        dataset="CIFAR10",
        train=False,
        transform=transform_test,
        backdoor=False,
    )

    test_dataset_backdoored = BackdooredDataset(
        root="../data",
        dataset="CIFAR10",
        train=False,
        transform=transform_test,
        backdoor=True,
        mode="replace",
        label_mode="clean_label",
        trigger_fn=gaussian_noise_static_trigger,
        p=1,
    )

    test_dataloader = DataLoader(
        test_dataset, Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
    )

    test_dataloader_backdoored = DataLoader(
        test_dataset_backdoored,
        Config.BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    return test_dataloader, test_dataloader_backdoored

In [6]:
def test(model, dataloader):
    predictions = []
    true_predictions = []

    model.eval()
    with torch.no_grad():
        for index, (images, labels) in enumerate(dataloader):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            true_predictions.extend(labels.cpu().numpy())
            predictions.extend(preds.cpu().numpy())

    return predictions, true_predictions

In [7]:
def plt_cm(model, data_loader, title, filename):
    predictions, true_predictions = test(model, data_loader)

    confusion_mx = confusion_matrix(
        y_pred=predictions, y_true=true_predictions, normalize="true"
    )

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        confusion_mx,
        annot=True,
        fmt=".2f",
        cmap="YlOrBr",
        xticklabels=CLASSES,
        yticklabels=CLASSES,
    )
    plt.xlabel("Prediction")
    plt.ylabel("True label")
    plt.title(title)
    plt.savefig(f"../images/{filename}", bbox_inches="tight")
    plt.close()

In [8]:
model = get_model()
clean_loader, backdoored_loader = get_data_loaders()

plt_cm(
    model,
    clean_loader,
    "Confusion Matrix on clean images on CIFAR-10",
    "plt_cm_on_cifar10_after_tl_cifar100_clean-2.png",
)
plt_cm(
    model,
    backdoored_loader,
    "Confusion Matrix on backdoored images on CIFAR-10",
    "plt_cm_on_cifar10_after_tl_cifar100_gaussian_static_100-0-2.png",
)